In [2]:
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from sacrebleu.metrics import BLEU, CHRF
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
# ==============================
# 3. Load your parallel dataset
# ==============================

src_file = "/kaggle/working/ja-ar.ja"
tgt_file = "/kaggle/working/ja-ar.ar"

ja_lines = []
ar_lines = []

with open(src_file, encoding="utf-8") as f_src, open(tgt_file, encoding="utf-8") as f_trg:

    for ja_line, ar_line in tqdm(zip(f_src, f_trg)):
        ja_line = ja_line.strip()
        ar_line = ar_line.strip()

        if ja_line and ar_line:
            ja_lines.append(ja_line)
            ar_lines.append(ar_line)

assert len(ja_lines) == len(ar_lines)

print("Total sentence pairs:", len(ja_lines))






3758064it [00:05, 637785.98it/s]

Total sentence pairs: 3758064


In [4]:
# ==============================
# 4. Create dataset
# ==============================

dataset = Dataset.from_dict({
    "ja": ja_lines,
    "ar": ar_lines
})

In [5]:
# ==============================
# 5. Train / validation split
# ==============================
dataset = dataset.train_test_split(test_size=0.1)

dataset["train"] = dataset["train"].shuffle().select(range(300000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(10000))


dataset_dict = {
    "train": dataset["train"],
    "validation": dataset["test"]
}


print(dataset)

DatasetDict({
    train: Dataset({
        features: ['ja', 'ar'],
        num_rows: 300000
    })
    test: Dataset({
        features: ['ja', 'ar'],
        num_rows: 10000
    })
})


In [ ]:

print("First example:")
for i in range(5):
   print(dataset["train"][i])

print("\nSample sizes:")
print({split: len(ds) for split, ds in dataset.items()})

In [6]:

model_name = "/kaggle/working/final_model3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
# ────────────────────────────────────────────────
# 3. Preprocessing function
# ────────────────────────────────────────────────

def preprocess(examples):
    model_inputs = tokenizer(examples["ja"], max_length=132, truncation=True, padding="max_length")
    
    labels = tokenizer(examples["ar"], max_length=132, truncation=True, padding="max_length")["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs






In [8]:

tokenized_datasets = dataset.map(
    preprocess,
    batched=True,
   
)


Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets["train"]
print("\nSample sizes:")
print({split: len(ds) for split, ds in tokenized_datasets.items()})

In [9]:
from transformers import Seq2SeqTrainingArguments 
# ────────────────────────────────────────────────
# 4. Training arguments (conservative settings)
# ────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir="./opus-ja-ar",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_steps=5000,
    label_smoothing_factor=0.1,
    eval_strategy="steps",
    eval_steps=5000,
    save_steps=5000,
    logging_steps=500,
    save_total_limit=3,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=True,
    report_to="none"
)



In [11]:
# ==============================
# 9. Evaluation metrics
# ==============================

bleu = BLEU()
chrf = CHRF()

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu_score = bleu.corpus_score(decoded_preds, [decoded_labels]).score
    chrf_score = chrf.corpus_score(decoded_preds, [decoded_labels]).score

    return {
        "bleu": bleu_score,
        "chrf": chrf_score
    }


In [12]:


# ==============================
# 8. Data collator
# ==============================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_datasets["train"],

    eval_dataset=tokenized_datasets["test"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)




In [ ]:
import torch
import gc

# 1. Clear everything possible
torch.cuda.empty_cache()
gc.collect()

# 2. Enable PyTorch's expandable segments (reduces fragmentation)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3. Print memory status again
!nvidia-smi
print("Allocated after clear:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved after clear:", torch.cuda.memory_reserved() / 1024**3, "GB")

In [13]:
print("Train size:", len(tokenized_datasets["train"]))
print("Validation size:", len(tokenized_datasets["validation"]))

Train size: 300000


KeyError: 'validation'

In [14]:


# ────────────────────────────────────────────────
# 6. Start training
# ────────────────────────────────────────────────
trainer.train()


Step,Training Loss,Validation Loss,Bleu,Chrf
5000,12.953595,1.617291,9.550632,23.445078
10000,12.877047,1.612079,10.251469,25.117411


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mtime may not be reliable on this filesystem, falling back to numerical ordering
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mtime may not be reliable on this filesystem, falling back to numerical ordering


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mtime may not be reliable on this filesystem, falling back to numerical ordering


TrainOutput(global_step=14064, training_loss=12.935248447630864, metrics={'train_runtime': 24488.512, 'train_samples_per_second': 36.752, 'train_steps_per_second': 0.574, 'total_flos': 3.14619199488e+16, 'train_loss': 12.935248447630864, 'epoch': 3.0})

In [ ]:
final_results = trainer.predict(tokenized_datasets["test"])
print("Final test results:")
print(final_results.metrics)

In [ ]:
trainer.save_model("/kaggle/working/final_model3")
tokenizer.save_pretrained("/kaggle/working/final_model3")

!zip -r final_model3.zip /kaggle/working/final_model3

from IPython.display import FileLink
FileLink(r'final_model3.zip')

In [ ]:
# ========================================
# BLEU Test using your custom function
# ========================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu
from tqdm.auto import tqdm

# ==================== LOAD MODEL ====================
model_path = "/kaggle/working/final_model3"   # ← Change this if your folder name is different

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

print("✅ Model loaded successfully!\n")

# ==================== YOUR CUSTOM TRANSLATION FUNCTION ====================
def translate_ja_to_ar_opus(text):
    # Add prefix if needed for Opus-MT
    if not text.startswith(">>jpn<<"):
        text = f">>jpn<< {text}"
    
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    
    translated_ids = model.generate(
        **inputs,
        max_length=200,
        num_beams=4,
        early_stopping=True
    )
    
    return tokenizer.decode(translated_ids[0], skip_special_tokens=True).strip()

# ==================== SIMPLE BLEU TEST ====================
def simple_bleu_test(dataset, num_examples=1000):
    references = []
    hypotheses = []
    
    print(f"Testing on {num_examples} examples...")
    
    for i in tqdm(range(min(num_examples, len(dataset)))):
        example = dataset[i]
        ja_text = example["ja"]
        ar_ref = example["ar"]
        
        # Use your custom function
        output = translate_ja_to_ar_opus(ja_text)
        
        hypotheses.append(output)
        references.append([ar_ref])
    
    # Calculate BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, references)
    
    print("\n=== BLEU SCORE ===")
    print(f"BLEU  : {bleu.score:.2f}")
    print(f"BP    : {bleu.bp:.3f}")
    print(f"Ratio : {bleu.sys_len / bleu.ref_len:.3f}")
    
    return bleu.score

# ==================== RUN THE TEST ====================
bleu_score = simple_bleu_test(dataset["test"], num_examples=1000)

print(f"\nFinal BLEU Score: {bleu_score:.2f}")



In [ ]:
print("\n=== MANUAL TEST WITH PREFIX ===\n")

test_sentences = [
    "やめてくれ！ 怖いんだよ！！",
    "このマンガ超面白いね！！",
    "うわっ！ 熱っ！ 熱すぎるって！！",
    "好きだよ…ずっと前から…",
    "あははははっ！ やられたぁ！",
]

for ja in test_sentences:
    ar = translate_ja_to_ar_opus(ja)
    print(f"JA: {ja}")
    print(f"AR: {ar}")
    print("-" * 60)